# 🔥 Colz GPT — Colab Training Notebook
**Run All → Done.**

| Step | Kya hoga |
|------|----------|
| 1 | GPU check |
| 2 | GitHub se code + dataset.jsonl |
| 3 | transformer / trainer / inference overwrite |
| 4 | JSONL → Corpus → Tokenize |
| 5 | Model build + torch.compile |
| 6 | Train (autocast fp16 + batch 256) |
| 7 | Generation test |
| 8 | brain.pt + vocab.json download |

In [ ]:
# ── CELL 1 : GPU CHECK ───────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {name}  |  VRAM: {mem:.1f} GB')
    print(f'   PyTorch : {torch.__version__}')
    print(f'   CUDA    : {torch.version.cuda}')
else:
    print('❌ GPU nahi mila!')
    print('   Runtime → Change runtime type → T4 GPU → Save → Run All')

In [ ]:
# ── CELL 2 : GITHUB CLONE / PULL ─────────────────────────────────────────
import os

GITHUB_REPO = 'https://github.com/niteenmaurya/Colz.git'
FOLDER      = '/content/gpt'

if os.path.exists(FOLDER):
    print('Folder pehle se hai — pulling latest...')
    os.system(f'cd {FOLDER} && git reset --hard HEAD && git pull')
else:
    os.system(f'git clone {GITHUB_REPO} {FOLDER}')

os.chdir(FOLDER)
print(f'\n📁 Workspace : {os.getcwd()}')
print('📁 Files     :', '  '.join(sorted(os.listdir())))

In [ ]:
# ── CELL 3 : WRITE transformer.py ────────────────────────────────────────
transformer_code = '''
import torch, math
import torch.nn as nn

DEVICE = (
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cpu")
)
print(f"[Transformer] device: {DEVICE}")

class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff, max_seq_len):
        super().__init__()
        self.vocab_size  = vocab_size
        self.d_model     = d_model
        self.max_seq_len = max_seq_len
        self.token_emb   = nn.Embedding(vocab_size, d_model)
        self.pos_emb     = nn.Embedding(max_seq_len, d_model)
        self.blocks      = nn.ModuleList([
            _TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)
        ])
        self.ln_final    = nn.LayerNorm(d_model)
        self.lm_head     = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)
        self.to(DEVICE)
        self._last_logits_tensor = None

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, token_ids):
        is_list = isinstance(token_ids, list)
        ids = torch.tensor(token_ids, dtype=torch.long, device=DEVICE) if is_list else token_ids.to(DEVICE)
        if ids.dim() == 1: ids = ids.unsqueeze(0)
        B, T = ids.shape
        if T > self.max_seq_len: ids = ids[:, -self.max_seq_len:]; T = self.max_seq_len
        pos    = torch.arange(T, device=DEVICE).unsqueeze(0)
        x      = self.token_emb(ids) + self.pos_emb(pos)
        for blk in self.blocks: x = blk(x)
        logits = self.lm_head(self.ln_final(x))
        self._last_logits_tensor = logits
        return logits[0].tolist() if is_list else logits

    def backward(self, dlogits): pass
    def zero_grad(self):
        for p in self.parameters():
            if p.grad is not None: p.grad.detach_(); p.grad.zero_()
    def all_params_and_grads(self):
        r = []
        for name, param in self.named_parameters():
            g  = param.grad
            gl = g.tolist() if g is not None else [[0.0]*param.shape[-1]]*(param.shape[0] if param.dim()>1 else 1)
            r.append((param.tolist(), gl, name))
        return r
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())


class _TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = _CausalSelfAttention(d_model, n_heads)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = _FeedForward(d_model, d_ff)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class _CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads  = n_heads
        self.d_head   = d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3*d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model,   bias=False)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv_proj(x).chunk(3, dim=-1)
        def rsh(t): return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        q, k, v = rsh(q), rsh(k), rsh(v)
        scores  = torch.matmul(q, k.transpose(-2,-1)) / math.sqrt(self.d_head)
        mask    = torch.triu(torch.full((T,T), float("-inf"), device=x.device), diagonal=1)
        attn    = torch.softmax(scores + mask, dim=-1)
        out     = torch.matmul(attn, v).transpose(1,2).contiguous().view(B, T, C)
        return self.out_proj(out)


class _FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model,d_ff), nn.GELU(), nn.Linear(d_ff,d_model))
    def forward(self, x): return self.net(x)
'''
with open('transformer.py', 'w', encoding='utf-8') as f: f.write(transformer_code)
print('✅ transformer.py written')

In [ ]:
# ── CELL 4 : WRITE trainer.py  (autocast + GradScaler + batch 256) ───────
trainer_code = '''
import math, random, time
import torch, torch.nn as nn, torch.optim as optim

DEVICE = (
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cpu")
)

class AdamOptimizer:
    def __init__(self, lr=3e-4, **kw): self.t = 0
    def step(self, p): pass

def clip_gradients(p, max_norm=1.0): return max_norm
def cross_entropy_loss(l, t): return 0.0, []


class Trainer:
    def __init__(self, model, lr=3e-4, max_norm=1.0, batch_size=256):
        self.model      = model
        self.max_norm   = max_norm
        self.batch_size = batch_size
        self.torch_optimizer = optim.AdamW(
            model.parameters(), lr=lr,
            betas=(0.9,0.999), eps=1e-8, weight_decay=0.01
        )
        self.optimizer    = AdamOptimizer(lr=lr)
        self._use_amp     = (DEVICE.type == "cuda")
        self.scaler       = torch.cuda.amp.GradScaler(enabled=self._use_amp)
        self.loss_history = []
        self._loss_fn     = nn.CrossEntropyLoss(ignore_index=0)

    def train_step(self, batch_windows, seq_len):
        self.torch_optimizer.zero_grad()
        IL, TL = [], []
        for tids in batch_windows:
            if len(tids) < 2: continue
            inp, tgt = tids[:-1], tids[1:]
            if len(inp) > seq_len: inp, tgt = inp[-seq_len:], tgt[-seq_len:]
            pad = seq_len - len(inp)
            if pad > 0: inp += [0]*pad; tgt += [0]*pad
            IL.append(inp); TL.append(tgt)
        if not IL: return 0.0
        inp_t = torch.tensor(IL, dtype=torch.long, device=DEVICE)
        tgt_t = torch.tensor(TL, dtype=torch.long, device=DEVICE)
        B, T  = inp_t.shape
        try:
            with torch.autocast(device_type=DEVICE.type, enabled=self._use_amp):
                pos    = torch.arange(T, device=DEVICE).unsqueeze(0)
                x      = self.model.token_emb(inp_t) + self.model.pos_emb(pos)
                for blk in self.model.blocks: x = blk(x)
                logits = self.model.lm_head(self.model.ln_final(x))
                loss   = self._loss_fn(logits.view(-1, logits.size(-1)), tgt_t.view(-1))
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.torch_optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.max_norm)
            self.scaler.step(self.torch_optimizer)
            self.scaler.update()
            return loss.item()
        except Exception as e:
            print(f"⚠️ train_step: {e}"); return 0.0

    def train(self, all_token_ids, seq_len, epochs, log_every=5):
        windows = [all_token_ids[s:s+seq_len+1]
                   for s in range(0, len(all_token_ids)-seq_len, seq_len)]
        if not windows: windows = [all_token_ids]
        sched = optim.lr_scheduler.CosineAnnealingLR(
            self.torch_optimizer, T_max=epochs, eta_min=1e-5)
        print("="*65)
        print(f"🔥  autocast={'ON fp16' if self._use_amp else 'OFF'}  device={DEVICE.type.upper()}")
        print(f"👉  batch={self.batch_size}  params={self.model.count_parameters():,}  windows={len(windows)}")
        print("="*65)
        self.model.train(); t0 = time.time()
        for epoch in range(1, epochs+1):
            te = time.time()
            random.shuffle(windows)
            batches = [windows[i:i+self.batch_size] for i in range(0,len(windows),self.batch_size)]
            el, sr = 0.0, 0
            for b in batches: el += self.train_step(b, seq_len); sr += 1
            avg = el / max(sr,1)
            self.loss_history.append(avg)
            sched.step()
            if epoch % log_every == 0 or epoch == 1:
                ppl = math.exp(min(avg,20))
                lr  = self.torch_optimizer.param_groups[0]["lr"]
                print(f" 🌟 {epoch:4d}/{epochs} | loss={avg:.4f} | ppl={ppl:.2f} | lr={lr:.2e} | {time.time()-te:.2f}s")
        print(f"✅ Done in {(time.time()-t0)/60:.2f} min")
        self.model.eval()
'''
with open('trainer.py', 'w', encoding='utf-8') as f: f.write(trainer_code)
print('✅ trainer.py written  (autocast + GradScaler + batch_size=256)')

In [ ]:
# ── CELL 5 : WRITE inference.py  (full GPU, no Python loops) ─────────────
inference_code = '''
import torch
import torch.nn.functional as F

DEVICE = (
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cpu")
)

@torch.no_grad()
def _last_logits(model, ids):
    if ids.shape[1] > model.max_seq_len: ids = ids[:, -model.max_seq_len:]
    pos    = torch.arange(ids.shape[1], device=DEVICE).unsqueeze(0)
    x      = model.token_emb(ids) + model.pos_emb(pos)
    for blk in model.blocks: x = blk(x)
    return model.lm_head(model.ln_final(x))[0, -1, :]

def _top_k_filter(logits, k):
    if k <= 0 or k >= logits.size(-1): return logits
    return logits.masked_fill(logits < torch.topk(logits, k).values[-1], float("-inf"))

@torch.no_grad()
def _loop(model, tokenizer, prompt, max_new, temperature, top_k, use_greedy):
    model.eval()
    ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=DEVICE)
    out = []
    for _ in range(max_new):
        logits = _last_logits(model, ids)
        if use_greedy:
            nid = logits.argmax().item()
        else:
            if top_k > 0: logits = _top_k_filter(logits, top_k)
            probs = F.softmax(logits / max(temperature, 1e-6), dim=-1)
            nid   = torch.multinomial(probs, 1).item()
        if nid == tokenizer.EOS_ID: break
        out.append(nid)
        ids = torch.cat([ids, torch.tensor([[nid]], dtype=torch.long, device=DEVICE)], dim=1)
    return prompt + tokenizer.decode(out)

def greedy(model, tok, prompt, max_new=100, max_seq_len=128, stop_on_eos=True):
    return _loop(model, tok, prompt, max_new, 1.0, 0, True)

def sample(model, tok, prompt, max_new=100, temperature=0.8, max_seq_len=128, stop_on_eos=True):
    return _loop(model, tok, prompt, max_new, temperature, 0, False)

def top_k_sample(model, tok, prompt, max_new=100, k=10, temperature=0.8, max_seq_len=128, stop_on_eos=True):
    return _loop(model, tok, prompt, max_new, temperature, k, False)

def generate(model, tok, prompt, max_new=100, strategy="top_k", temperature=0.8, k=10, max_seq_len=128):
    s = strategy.lower()
    if s == "greedy":           return greedy(model, tok, prompt, max_new)
    elif s == "sample":         return sample(model, tok, prompt, max_new, temperature)
    elif s in ("top_k","topk"): return top_k_sample(model, tok, prompt, max_new, k, temperature)
    else: raise ValueError(f"Unknown strategy: {strategy}")
'''
with open('inference.py', 'w', encoding='utf-8') as f: f.write(inference_code)
print('✅ inference.py written  (GPU tensor loop, no .tolist() overhead)')

In [ ]:
# ── CELL 6 : JSONL → CORPUS (NO data.txt step) ───────────────────────────
import json, os

SYSTEM_PROMPT = "You are a friendly, helpful, and witty AI assistant."
JSONL_FILE    = "dataset.jsonl"

if not os.path.exists(JSONL_FILE):
    raise FileNotFoundError(f"❌ {JSONL_FILE} not found. Push it to GitHub first.")

corpus_pieces = []
skipped = 0

with open(JSONL_FILE, 'r', encoding='utf-8') as f:
    for ln, line in enumerate(f, 1):
        line = line.strip()
        if not line: continue
        try:
            data = json.loads(line)
            msgs = data.get('messages', [])
            if not msgs: continue
            # System prompt inject karo har conversation ke upar
            corpus_pieces.append(f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n")
            for msg in msgs:
                role    = msg.get('role', '').strip().lower()
                content = msg.get('content', '').strip()
                if role and content:
                    corpus_pieces.append(f"<|im_start|>{role}\n{content}<|im_end|>\n")
        except json.JSONDecodeError:
            skipped += 1

CORPUS = ''.join(corpus_pieces)

print(f'✅ dataset.jsonl padha')
print(f'   Conversations : {len([p for p in corpus_pieces if "system" in p])}')
print(f'   Corpus size   : {len(CORPUS):,} chars')
if skipped: print(f'   ⚠️ Skipped {skipped} malformed lines')

In [ ]:
# ── CELL 7 : TOKENIZER ───────────────────────────────────────────────────
import sys, importlib

sys.path.insert(0, os.getcwd())
for mod in ['tokenizer', 'transformer', 'trainer', 'inference']:
    if mod in sys.modules: importlib.reload(sys.modules[mod])

from tokenizer import CharTokenizer

tok = CharTokenizer()
tok.build_vocab(CORPUS)      # seedha CORPUS se — koi file nahi
tok.save('vocab.json')
all_ids = tok.encode(CORPUS)

print(f'✅ Vocab size : {tok.vocab_size}')
print(f'   Token IDs  : {len(all_ids):,}')

In [ ]:
# ── CELL 8 : MODEL + torch.compile ───────────────────────────────────────
import torch._dynamo
from transformer import GPT

# ─── CONFIG — sirf yahan badlo ──────────────────────────────────────────
CFG = dict(
    vocab_size  = tok.vocab_size,
    d_model     = 256,
    n_heads     = 8,
    n_layers    = 6,
    d_ff        = 1024,
    max_seq_len = 128,
)
TRAIN_CFG = dict(
    lr         = 3e-4,
    batch_size = 256,
    epochs     = 2000,
    log_every  = 25,
)
# ─────────────────────────────────────────────────────────────────────────

model = GPT(**CFG)
print(f'✅ Model params : {model.count_parameters():,}')

torch._dynamo.config.suppress_errors = True
model = torch.compile(model)
print('✅ torch.compile active  (~1.5-2x extra speed on T4)')

In [ ]:
# ── CELL 9 : TRAIN ───────────────────────────────────────────────────────
from trainer import Trainer

trainer = Trainer(
    model,
    lr         = TRAIN_CFG['lr'],
    batch_size = TRAIN_CFG['batch_size'],
)
trainer.train(
    all_ids,
    seq_len   = CFG['max_seq_len'],
    epochs    = TRAIN_CFG['epochs'],
    log_every = TRAIN_CFG['log_every'],
)

In [ ]:
# ── CELL 10 : GENERATION TEST ────────────────────────────────────────────
from inference import generate

SYS = "You are a friendly, helpful, and witty AI assistant."

def make_prompt(user_msg):
    return (f"<|im_start|>system\n{SYS}<|im_end|>\n"
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n")

test_msgs = [
    "Hi!",
    "Kya chal raha hai?",
    "Tell me a joke.",
    "What's your favorite holiday?",
]

print('='*60)
print('🤖 GENERATION TEST')
print('='*60)

for msg in test_msgs:
    p     = make_prompt(msg)
    out   = generate(model, tok, p, max_new=80, strategy='top_k', k=5, temperature=0.8)
    reply = out[len(p):].split('<|im_end|>')[0].strip()
    print(f'User  : {msg}')
    print(f'Bot   : {reply}')
    print('-'*60)

In [ ]:
# ── CELL 11 : SAVE + LOSS CURVE + DOWNLOAD ───────────────────────────────
import pickle
import matplotlib.pyplot as plt
from google.colab import files

# brain.pt
torch.save({
    'model_state_dict' : model.state_dict(),
    'config'           : CFG,
    'loss_history'     : trainer.loss_history,
}, 'brain.pt')

# brain.pkl backup
with open('brain.pkl', 'wb') as f:
    pickle.dump(model, f)

# Loss curve
plt.figure(figsize=(10, 4))
plt.plot(trainer.loss_history, color='crimson', linewidth=1.5)
plt.title('Training Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=120)
plt.show()

print('✅ Saved: brain.pt | vocab.json | loss_curve.png')
print('⬇️  Downloading...')

files.download('brain.pt')
files.download('vocab.json')
files.download('loss_curve.png')

print('🚀 DONE!')